# P05 — Odziv sustava drugog reda: klizač za faktor prigušenja ζ

Bilježnica uz predavanje P05 i **poglavlje 5 udžbenika *Signali i sustavi*** (Sustavi prvog i
drugog reda). Normirani sustav drugog reda pobuđujemo jediničnim skokom i klizačem mijenjamo
faktor prigušenja $\zeta$ od $0$ do $2$. Uz odziv se crtaju polovi u s-ravnini: za $\zeta<1$
gibaju se po kružnici polumjera $\omega_n$, a za $\zeta>1$ po realnoj osi. Bilježnica ispisuje
režim prigušenja i pokazatelje odziva: vršno vrijeme $t_p$, nadvišenje $M_p$ i vrijeme
smirivanja $t_s$.

**Kako pokrenuti.** Ćelije pokrenite redom (Shift + Enter) ili izbornikom *Run → Run All
Cells*. Instalaciju Pythona i JupyterLaba opisuje `README.md` repozitorija. Bilježnica radi bez
interneta i nema zvuka, pa ne trebaju ni zvučnici. Ako se klizač ne prikaže, funkciju pokusa
pozovite izravno (odjeljak 4).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from scipy.optimize import brentq
from IPython.display import display
import ipywidgets as widgets

BOJA_ZETA = "#840B55"                  # bordo: odabrani ζ na klizaču
SIVA, TAMNA = "#5B6770", "#222222"
# Krivulje se razlikuju vrstom crte i oznakom, pa su čitljive i crno-bijelo.
CRTE = ["-", "--", "-.", ":", (0, (7, 2, 1, 2, 1, 2))]
OZNAKE = ["o", "s", "^", "D", "v"]

# decimalni zarez i pravi minus na osima
zarez = FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",").replace("-", "−"))
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.3})

## Postavke

Sustav je **normiran**: vlastita frekvencija je $\omega_n = 1$ rad/s. Drugi $\omega_n$ zadajte u
ćeliji ispod, pa pokrenite sve ćelije.

In [ ]:
# ===== POSTAVKE (mijenjajte po želji, pa Run → Run All Cells) =====
WN = 1.0                 # rad/s: vlastita frekvencija ω_n
ZETA_KORAK = 0.05        # korak klizača ζ
ZETA_MAX = 2.0           # najveći ζ na klizaču (najmanji je 0)
ZETA_POCETNA = 0.3       # ζ na klizaču nakon pokretanja
ZETE_USPOREDBA = (0.2, 0.5, 0.7, 1.0, 2.0)   # nekoliko fiksnih ζ za usporedbu
T_KRAJ = 20 / WN         # s: vremenska os odziva

## 1. Normirani sustav drugog reda i polovi

Sustav drugog reda u **normiranom obliku** (poglavlje 5) iz mirovanja, $x(0)=\dot{x}(0)=0$,
pobuđujemo jediničnim skokom:

$$\ddot{x} + 2\zeta\omega_n\,\dot{x} + \omega_n^2\,x = \omega_n^2\,x_\infty ,\qquad t\ge 0,\qquad x_\infty = 1 .$$

Isti sustav opisuje prijenosna funkcija (poglavlje 11)
$G(s) = \dfrac{\omega_n^2}{s^2 + 2\zeta\omega_n s + \omega_n^2}$. Korijeni karakteristične
jednadžbe $s^2 + 2\zeta\omega_n s + \omega_n^2 = 0$ su **polovi**

$$s_{1,2} = -\zeta\omega_n \pm \omega_n\sqrt{\zeta^2 - 1} .$$

**Režimi prigušenja:**

| $\zeta$ | režim | polovi | odziv na skok |
|---|---|---|---|
| $0$ | neprigušen | $\pm j\omega_n$, na imaginarnoj osi | titra oko $x_\infty$ bez kraja |
| $0<\zeta<1$ | potprigušen | $-\sigma \pm j\omega_d$, na kružnici polumjera $\omega_n$ | prebaci $x_\infty$ i titra sve slabije |
| $1$ | kritično prigušen | dvostruki $-\omega_n$ | bez nadvišenja, najbrži od takvih |
| $\zeta>1$ | natprigušen | dva realna, $s_1 s_2 = \omega_n^2$ | bez nadvišenja, sporiji |

Za potprigušen sustav je $\sigma = \zeta\omega_n$, $\omega_d = \omega_n\sqrt{1-\zeta^2}$ i
$|s_{1,2}| = \omega_n$, a kut pola prema negativnoj realnoj osi je $\theta = \arccos\zeta$.

Funkcije u ćeliji ispod računaju polove, odziv (zatvoreni izrazi iz poglavlja 5, za sva tri
režima) i pokazatelje. **Stvarno vrijeme smirivanja** $t_s$ je trenutak zadnjeg ulaska odziva u
pojas $\pm 2\,\%$ oko $x_\infty$, a $4/(\zeta\omega_n)$ je samo njegova procjena.

In [ ]:
POJAS = 0.02             # pojas vremena smirivanja: ±2 % oko x_∞ (poglavlje 5)


def broj(v, decimala=2):
    """Broj s decimalnim zarezom i pravim minusom (−0,50); −0,00 piše se 0,00."""
    tekst = f"{v:.{decimala}f}"
    if float(tekst) == 0:
        tekst = tekst.lstrip("-")
    return tekst.replace(".", ",").replace("-", "−")


def polovi(zeta, wn=WN):
    """Polovi s₁ i s₂, korijeni s² + 2ζω_n·s + ω_n² = 0 (kompleksni brojevi)."""
    d = np.sqrt(complex(zeta**2 - 1))
    return -zeta * wn + wn * d, -zeta * wn - wn * d


def kriticno(zeta):
    return bool(np.isclose(zeta, 1))


def rezim(zeta):
    """Naziv režima prigušenja."""
    if zeta == 0:
        return "neprigušen sustav (ζ = 0)"
    if kriticno(zeta):
        return "kritično prigušen sustav (ζ = 1)"
    return "potprigušen sustav (0 < ζ < 1)" if zeta < 1 else "natprigušen sustav (ζ > 1)"


def odziv(t, zeta, wn=WN):
    """Odziv x(t) na jedinični skok iz mirovanja, x(0) = ẋ(0) = 0, x_∞ = 1."""
    t = np.asarray(t, dtype=float)
    if kriticno(zeta):
        return 1 - (1 + wn * t) * np.exp(-wn * t)
    if zeta < 1:
        sigma, wd = zeta * wn, wn * np.sqrt(1 - zeta**2)
        return 1 - np.exp(-sigma * t) * (np.cos(wd * t) + sigma / wd * np.sin(wd * t))
    s1, s2 = (s.real for s in polovi(zeta, wn))
    return 1 - (s2 * np.exp(s1 * t) - s1 * np.exp(s2 * t)) / (s2 - s1)


def stvarno_ts(zeta, wn=WN, pojas=POJAS):
    """Zadnji ulazak odziva u pojas ±pojas oko x_∞ = 1 (None za ζ = 0: titranje se ne gasi)."""
    if zeta == 0:
        return None
    if zeta < 1 and not kriticno(zeta):
        # Kad ovojnica uđe u pojas, odziv iz njega više ne izlazi: dovoljno je gledati do tada.
        kraj = np.log(1 / (pojas * np.sqrt(1 - zeta**2))) / (zeta * wn) * 1.001
        t = np.linspace(0, kraj, 200001)
        i = np.flatnonzero(np.abs(odziv(t, zeta, wn) - 1) > pojas)[-1]
        return brentq(lambda u: abs(odziv(u, zeta, wn) - 1) - pojas, t[i], t[i + 1])
    spori = -polovi(zeta, wn)[0].real          # pol bliži ishodištu određuje brzinu
    return brentq(lambda u: odziv(u, zeta, wn) - (1 - pojas), 0, 50 / spori)


def pokazatelji(zeta, wn=WN):
    """t_p, M_p (omjer), procjena 4/(ζω_n) i stvarno t_s; None gdje pokazatelja nema."""
    p = {"t_p": None, "M_p": None, "ts_procjena": None, "t_s": stvarno_ts(zeta, wn)}
    if zeta < 1 and not kriticno(zeta):
        p["t_p"] = np.pi / (wn * np.sqrt(1 - zeta**2))                # t_p = π/ω_d
        p["M_p"] = np.exp(-np.pi * zeta / np.sqrt(1 - zeta**2))
        if zeta > 0:
            p["ts_procjena"] = 4 / (zeta * wn)
    return p


def kratko(v):
    """Kratak zapis broja za legende: 0,2; 1; 2."""
    return f"{v:g}".replace(".", ",")


def postotak(mp):
    """Nadvišenje u postocima: jedna decimala, ispod 1 % dvije."""
    if 100 * mp < 0.005:
        return "manje od 0,01 %"
    return f"{broj(100 * mp, 1 if 100 * mp >= 1 else 2)} %"


def opis(zeta, wn=WN):
    """Recci ispisa za odabrani ζ: režim, polovi i pokazatelji."""
    s1, s2 = polovi(zeta, wn)
    p = pokazatelji(zeta, wn)
    recci = [f"ζ = {broj(zeta)}, ω_n = {broj(wn)} rad/s: {rezim(zeta)}"]
    if zeta == 0:
        recci.append(f"polovi s₁,₂ = ±j{broj(wn)} na imaginarnoj osi (σ = 0)")
    elif zeta < 1 and not kriticno(zeta):
        recci.append(f"polovi s₁,₂ = {broj(s1.real)} ± j{broj(s1.imag)} na kružnici |s| = ω_n "
                     f"(σ = {broj(-s1.real)} 1/s, ω_d = {broj(s1.imag)} rad/s)")
    elif kriticno(zeta):
        recci.append(f"dvostruki realni pol s = −ω_n = {broj(-wn)}")
    else:
        recci.append(f"realni polovi s₁ = {broj(s1.real)} i s₂ = {broj(s2.real)} "
                     "(umnožak s₁·s₂ = ω_n²)")
    if p["t_p"] is None:
        recci.append("nadvišenja nema: odziv raste prema x_∞ = 1 bez prebačaja (nema ni t_p)")
    else:
        recci.append(f"vršno vrijeme t_p = π/ω_d = {broj(p['t_p'])} s, "
                     f"nadvišenje M_p = {postotak(p['M_p'])}")
    if p["t_s"] is None:
        recci.append("vremena smirivanja nema: odziv titra oko x_∞ = 1 bez kraja")
    elif p["ts_procjena"] is None:
        recci.append(f"stvarno vrijeme smirivanja (±2 %) t_s = {broj(p['t_s'])} s")
    else:
        recci.append(f"vrijeme smirivanja (±2 %): procjena 4/(ζω_n) = "
                     f"{broj(p['ts_procjena'])} s, stvarno t_s = {broj(p['t_s'])} s")
    return recci


for redak in opis(ZETE_USPOREDBA[1]):
    print(redak)

## 2. Familija odziva za nekoliko vrijednosti ζ

Odzivi za fiksne vrijednosti $\zeta$ iz postavki crtaju se na istom grafu. Osjenčan je pojas
$\pm 2\,\%$ oko $x_\infty = 1$. Što je $\zeta$ manji, to je nadvišenje veće i titranje dulje. Za
$\zeta \ge 1$ nadvišenja nema, a veći $\zeta$ daje sporiji odziv.

In [ ]:
t = np.linspace(0, T_KRAJ, 2001)
fig, ax = plt.subplots(figsize=(10, 3.8))
ax.axhspan(1 - POJAS, 1 + POJAS, color=SIVA, alpha=0.2, lw=0, label="pojas ±2 %")
for i, z in enumerate(ZETE_USPOREDBA):
    ax.plot(t, odziv(t, z), color=TAMNA, lw=1.5, ls=CRTE[i % len(CRTE)],
            label=f"ζ = {kratko(z)}")
ax.set_xlim(0, T_KRAJ)
ax.set_ylim(0, 1.8)
ax.set_xlabel(f"vrijeme $t$ / s   ($\\omega_n$ = {broj(WN)} rad/s)")
ax.set_ylabel("odziv $x(t)$")
ax.set_title("Odziv na jedinični skok za nekoliko vrijednosti $\\zeta$", loc="left", fontsize=10)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)   # izvan krivulja
ax.xaxis.set_major_formatter(zarez)
ax.yaxis.set_major_formatter(zarez)
fig.tight_layout()
plt.show()

## 3. Polovi u s-ravnini

Polovi za iste vrijednosti $\zeta$. Za $0 \le \zeta < 1$ leže na kružnici polumjera $\omega_n$
(crtkano), pod kutom $\theta = \arccos\zeta$ prema negativnoj realnoj osi. Za $\zeta = 1$
spoje se u dvostruki pol $-\omega_n$, a za $\zeta > 1$ razdvoje se po realnoj osi: jedan ide
prema ishodištu, drugi od njega, uz stalan umnožak $s_1 s_2 = \omega_n^2$. Siva crta je put
polova dok $\zeta$ raste od $0$ do najvećeg $\zeta$ na klizaču.

In [ ]:
def s_ravnina(ax):
    # kružnica polumjera ω_n, put polova za ζ od 0 do ZETA_MAX i osi s-ravnine
    ax.axhline(0, color="k", lw=0.6, zorder=1)
    ax.axvline(0, color="k", lw=0.6, zorder=1)
    fi = np.linspace(0, 2 * np.pi, 361)
    ax.plot(WN * np.cos(fi), WN * np.sin(fi), color=SIVA, ls="--", lw=0.8, zorder=2)
    put = dict(color=SIVA, lw=4, alpha=0.4, solid_capstyle="butt", zorder=2)
    z = np.linspace(0, 1, 201)
    for predznak in (1, -1):
        ax.plot(-z * WN, predznak * WN * np.sqrt(1 - z**2), **put)
    z = np.linspace(1, ZETA_MAX, 201)
    ax.plot(-WN * (z + np.sqrt(z**2 - 1)), 0 * z, **put)
    ax.plot(-WN * (z - np.sqrt(z**2 - 1)), 0 * z, **put)
    lijevo = -WN * (ZETA_MAX + np.sqrt(max(ZETA_MAX**2 - 1, 0))) * 1.1
    ax.set_xlim(lijevo, 0.6 * WN)
    ax.set_ylim(-1.45 * WN, 1.45 * WN)
    ax.set_aspect("equal")
    ax.set_xlabel("Re $s$")
    ax.set_ylabel("Im $s$")
    ax.xaxis.set_major_formatter(zarez)
    ax.yaxis.set_major_formatter(zarez)


fig, ax = plt.subplots(figsize=(10, 3.6))
s_ravnina(ax)
for i, z in enumerate(ZETE_USPOREDBA):
    s1, s2 = polovi(z)
    ax.plot([s1.real, s2.real], [s1.imag, s2.imag], OZNAKE[i % len(OZNAKE)], ms=8,
            mfc="white", mec=TAMNA, mew=1.5, label=f"ζ = {kratko(z)}")
ax.set_title("Polovi za nekoliko vrijednosti $\\zeta$ (crtkano: kružnica polumjera $\\omega_n$)",
             loc="left", fontsize=10)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)
fig.tight_layout()
plt.show()

## 4. Klizač: faktor prigušenja ζ

Pomaknite klizač $\zeta$ (od $0$ do $2$, korak $0{,}05$). Lijevo su polovi (×) na putu polova,
a desno odziv za odabrani $\zeta$ (puna crta), uz sive odzive za fiksne $\zeta$ iz odjeljka 2.
Kružić je vrh odziva u $t_p$, a točkasta okomita crta stvarno vrijeme smirivanja $t_s$, kad
stanu na graf. Ispod grafa su režim, polovi i pokazatelji.

Ako se klizač ne prikaže, pozovite funkciju izravno, npr. `pokus(zeta=0.6)`.

In [ ]:
def pokus(zeta=ZETA_POCETNA):
    zeta = round(zeta, 4)                       # klizač: 0.30000000000000004 -> 0.3
    p = pokazatelji(zeta)
    t = np.linspace(0, T_KRAJ, 2001)
    fig, (ax_s, ax_t) = plt.subplots(1, 2, figsize=(12, 4.2),
                                     gridspec_kw={"width_ratios": [1, 1.4]})
    s_ravnina(ax_s)
    s1, s2 = polovi(zeta)
    ax_s.plot([s1.real, s2.real], [s1.imag, s2.imag], "x", color=BOJA_ZETA, ms=12, mew=2.5)
    ax_s.set_title(f"polovi za $\\zeta$ = {broj(zeta)}", loc="left", fontsize=10)

    ax_t.axhspan(1 - POJAS, 1 + POJAS, color=SIVA, alpha=0.2, lw=0, label="pojas ±2 %")
    for i, z in enumerate(ZETE_USPOREDBA):
        ax_t.plot(t, odziv(t, z), color=SIVA, lw=0.9, ls=CRTE[i % len(CRTE)],
                  label=f"ζ = {kratko(z)}")
    ax_t.plot(t, odziv(t, zeta), color=BOJA_ZETA, lw=2.6, label=f"odabrani ζ = {broj(zeta)}")
    if p["t_p"] is not None and p["t_p"] <= T_KRAJ:
        ax_t.plot(p["t_p"], 1 + p["M_p"], "o", ms=8, mfc="white", mec=TAMNA, mew=1.5,
                  label="vrh u $t_p$")
    if p["t_s"] is not None and p["t_s"] <= T_KRAJ:
        ax_t.axvline(p["t_s"], color=TAMNA, ls=":", lw=1.5, label="stvarno $t_s$")
    ax_t.set_xlim(0, T_KRAJ)
    ax_t.set_ylim(-0.05, 2.05)
    ax_t.set_xlabel(f"vrijeme $t$ / s   ($\\omega_n$ = {broj(WN)} rad/s)")
    ax_t.set_ylabel("odziv $x(t)$")
    ax_t.set_title(rezim(zeta), loc="left", fontsize=10)
    ax_t.legend(loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False, fontsize=9)
    ax_t.xaxis.set_major_formatter(zarez)
    ax_t.yaxis.set_major_formatter(zarez)
    fig.tight_layout()
    plt.show()
    for redak in opis(zeta):
        print(redak)


klizac = widgets.interactive(
    pokus,
    zeta=widgets.FloatSlider(value=ZETA_POCETNA, min=0, max=ZETA_MAX, step=ZETA_KORAK,
                             description="ζ", readout_format=".2f", continuous_update=False),
)
display(klizac)

## 5. Veza s formulama poglavlja 5

Za potprigušen sustav ($0<\zeta<1$) poglavlje 5 izvodi

$$t_p = \frac{\pi}{\omega_d} = \frac{\pi}{\omega_n\sqrt{1-\zeta^2}},\qquad
M_p = e^{-\pi\zeta/\sqrt{1-\zeta^2}},\qquad t_s \approx \frac{4}{\zeta\omega_n}\ \ (\text{pojas } 2\,\%).$$

- **Pokazatelji u s-ravnini.** $t_p$ ovisi o visini pola $\omega_d$, procjena $t_s$ o
  udaljenosti pola od imaginarne osi $\sigma$, a $M_p$ o kutu $\theta = \arccos\zeta$.
- **Procjena i stvarno vrijeme smirivanja.** Kako je procjena $4/(\zeta\omega_n)$ izvedena i
  kada vrijedi, opisuje poglavlje 5 (odjeljak *Pokazatelji odziva na skok*).

Ćelija ispod provjerava formule na samoj krivulji odziva: vrh i njegov trenutak traži kao
najveći uzorak, a $t_s$ kao zadnji uzorak izvan pojasa. Uz to uspoređuje rezultat s tablicom
pokazatelja u ovisnosti o $\zeta$ iz poglavlja 5 (odjeljak *Pokazatelji odziva na skok*), u
kojoj su vremena normirana: $\omega_n t_p$ i $\sigma t_s = \zeta\omega_n t_s$. Ispisuju se
nadvišenje i vršno vrijeme; vrijeme smirivanja provjerava se bez ispisa.

In [ ]:
# Tablica iz knjige (poglavlje 5, Pokazatelji odziva na skok), za ω_n = 1 rad/s:
# ζ: (M_p u %, ω_n·t_p, σ·t_s za pojas 2 %)
TABLICA_KNJIGE = {
    0.1: (72.9, 3.16, 3.84),
    0.2: (52.7, 3.21, 3.92),
    0.3: (37.2, 3.29, 3.37),
    0.4: (25.4, 3.43, 3.36),
    0.5: (16.3, 3.63, 4.04),
    0.6: (9.5, 3.93, 3.57),
    0.7: (4.6, 4.40, 4.19),
    0.8: (1.5, 5.24, 3.00),
    0.9: (0.15, 7.21, 4.23),
}

t = np.linspace(0, 60, 600001)                  # korak 0,1 ms
print("   ζ    M_p (formula)   M_p (krivulja)   ω_n·t_p")
for z, (mp_knjiga, tpn_knjiga, tsn_knjiga) in TABLICA_KNJIGE.items():
    p = pokazatelji(z, wn=1.0)
    x = odziv(t, z, wn=1.0)
    i = int(np.argmax(x))                       # vrh: najveći uzorak
    assert abs((x[i] - 1) - p["M_p"]) < 1e-6 and abs(t[i] - p["t_p"]) < 1e-3
    izvan = np.flatnonzero(np.abs(x - 1) > POJAS)   # zadnji uzorak izvan pojasa
    assert abs(t[izvan[-1]] - p["t_s"]) < 1e-3
    d = 1 if mp_knjiga >= 1 else 2              # decimale kao u knjizi
    assert round(100 * p["M_p"], d) == mp_knjiga
    assert round(p["t_p"], 2) == tpn_knjiga and round(z * p["t_s"], 2) == tsn_knjiga
    print(f"{broj(z, 1):>5}   {broj(100 * p['M_p'], d):>9} %   {broj(100 * (x[i] - 1), d):>10} %"
          f"   {broj(p['t_p']):>7}")
print("Formule, krivulja odziva i tablica u knjizi daju iste pokazatelje.")

## Pokušajte sami

1. Nađite najmanji $\zeta$ na klizaču uz koji je nadvišenje najviše $5\,\%$. Provjerite ga
   formulom $\zeta = -\ln M_p \big/ \sqrt{\pi^2 + \ln^2 M_p}$ iz poglavlja 5.
2. Pomičite $\zeta$ od $0{,}6$ do $0{,}9$ i usporedite stvarno vrijeme smirivanja s procjenom
   $4/(\zeta\omega_n)$. Za koje $\zeta$ je stvarno vrijeme dulje od procjene? Što na grafu
   odziva tada određuje zadnji ulazak u pojas?
3. Predvidite iz formula za $t_p$, $M_p$ i $4/(\zeta\omega_n)$ što se s njima dogodi kad se
   $\omega_n$ udvostruči, a $\zeta$ ostane isti. Zatim u postavkama zadajte `WN = 2.0`,
   pokrenite sve ćelije i provjerite predviđanje na ispisu i na grafovima. Ponovite s
   `WN = 0.5`.
4. Usporedite $\zeta = 0{,}7$ i $\zeta = 1$. Koji odziv prvi dosegne $90\,\%$ konačne
   vrijednosti, a koji se prvi smiri u pojasu $\pm 2\,\%$? Zašto se u tehnici često bira
   $\zeta$ oko $0{,}7$?
5. Za $\zeta = 2$ dopišite ćeliju koja uz odziv crta i odziv sustava prvog reda
   $1 - e^{-t/\tau}$ s $\tau = 1/|s_1|$, gdje je $s_1$ pol bliži ishodištu. Koliko se dva
   odziva razlikuju i zašto?

---
Kod je pod licencom MIT (`LICENSE`), a tekst pod CC BY 4.0 (`LICENSE-sadrzaj.md`).